In [1]:
import gc

import env
import torch
from CADGNTrainer import CADGNTrainer, Stage1Config, Stage2Config

from losses import _generate_candidate_pairs
from models_config import models
from ds.cladder import CLadderSample, CLadderDataset, load_cladder_v1_5, CLadderLoaderConfig
from cadgn import CADGNCore, TokenizerFamily, flush_gpu, save_history, BaseEncoder, LLMWrapper
from search import ArchParams
from collections import defaultdict
import gc

# Unfortunately, the current implementation precludes saving modified versions of the various modules (config is static)

flush_gpu()
gc.collect()

epochs = 150

do_models = [ "Qwen/Qwen3-4B", "meta-llama/Llama-3.2-3B", "Qwen/Qwen3-8B-FP8", "meta-llama/Llama-3.1-8B"]

# rounded to 4dp where appropriate
best_params = {
    "ca_dgn_dim": 256,
    "max_layers": 5,
    "num_iters": 4,
    "epsilon": 0.0665, # 0.06646303815100524
    "base_gamma": 0.0296, # 0.02956587470709161
    "encoder_dropout": 0.0563,  # 0.05627028512612424
    "decoder_expansion": 4,
    "decoder_dropout": 0.1743,  # 0.17434218753510897
    "head_encoder_dropout": 0.0595,  # 0.05948165802005359
    "head_decoder_dropout": 0.0543,  # 0.0543472754267514
}

arch = ArchParams.from_dict(best_params)

s1c = Stage1Config(
    lr=0.0015,  # 0.0015121612751471838
    weight_decay=0.0004,  # 0.0004379467905083944
    grad_accum_steps=13,
    # Default
    max_steps_per_epoch=550,
    max_steps_per_epoch_val=150,
    w_mmd=1.0,
    epochs=150
)

s2c = Stage2Config(
    lr=0.0015,  # 0.0015121612751471838
    weight_decay=0.0004,  # 0.0004379467905083944
    grad_accum_steps=13,
    max_steps_per_epoch=550,
    max_steps_per_epoch_val=150,
    w_mmd = 1.0,
    epochs=150
)

evals2c = Stage2Config(
  **s2c.__dict__
)
evals2c.max_steps_per_epoch_val = None
evals2c.max_steps_per_epoch = None


"""
 Models to Ablate for:
    - LLM alone (prompt + gate cls)
    - GCN  (does the graph help?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False
    - ADGN (Gravina et al., 2023) (does Antisymmetry help?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False
    - CADGN (does a directed neighbourhood aggregation help?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False

    - CADGN (remove CADGN Decoder, is it necessary?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False

    - GATv2Conv (Brody et al., 2022) (does Antisymmetry + RoPE directionality help over learned attention?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False
    -

"""




GPU: 0.00GB allocated / 0.00GB reserved


'\n Models to Ablate for:\n    - LLM alone (prompt + gate cls)\n    - GCN  (does the graph help?)\n        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0\n        - Stage 2: graph_first=True/False\n    - ADGN (Gravina et al., 2023) (does Antisymmetry help?)\n        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0\n        - Stage 2: graph_first=True/False\n    - CADGN (does a directed neighbourhood aggregation help?)\n        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0\n        - Stage 2: graph_first=True/False\n\n    - CADGN (remove CADGN Decoder, is it necessary?)\n        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0\n        - Stage 2: graph_first=True/False\n\n    - GATv2Conv (Brody et al., 2022) (does Antisymmetry + RoPE directionality help over learned attention?)\n        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0\n        - Stage 2: graph_first=True/False\n    -\n\n'

In [2]:
dsConfig = CLadderLoaderConfig(rung_filter=None, query_types=None, skip_unparseable=True)

org_ds = load_cladder_v1_5(dsConfig)

train, vald = CLadderDataset.from_samples_split(
    samples=org_ds,
    val_size=0.2,
    stratify=True
)

train_d = train.as_dataloader(max_steps_per_epoch=550)
val_d = vald.as_dataloader(max_steps_per_epoch=150, shuffle=False)

complete_val = vald.as_dataloader()
prompt_samples: list[CLadderSample] = CLadderDataset(org_ds).stratified_sample(50)


Processing 10112 rows...
Loaded 8532 graphs, skipped 1580 unparseable rows.


In [3]:
import numpy as np
import pandas as pd

from cadgn import GateClassifier

from ablations.adgn_encoder import ADGNEncoder
from ablations.gat_encoder import GATEncoder
from ablations.gcn_encoder import GCNEncoder
from ablations.BaselineLLMTrainer import BaselineLLMTrainer, BaselineSampleResult
from cadgn.results import save_sample_results

def eval_and_save(
        llm: LLMWrapper,
        tokfam,
        mmd_val,
        history,
        metrics,
        per_sample,
        cls="baseline"
):

    safe_mdl_name = llm.model_id.replace("/", "__")
    save_history(history, path=f"./ablate_results/history/{cls}_{safe_mdl_name}_mmd-{int(mmd_val)}_history.csv")
    mdf = pd.DataFrame(metrics)
    mdf.to_csv(f"./ablate_results/metrics/{cls}_{safe_mdl_name}_mmd-{int(mmd_val)}_metrics.csv")
    save_sample_results(
        results=per_sample,
        path=f"./ablate_results/samples/",
        variant=f"{cls}_{safe_mdl_name}_mmd-{int(mmd_val)}",
        epoch=150
    )
    # Take a subset of prompts and get text output: see if the qualitative results match statistical expectations...
    print("\t>> Handling Qualitative Gather....")
    o_texts = []
    for sample in enumerate(prompt_samples):
        input_embeds = llm.embed_prompt(
            prompt=sample.prompt,
            tokenizer=tokfam.tokenizer,
            embed_layer=tokfam.embed_layer,
            device=llm.device
        )
        _, attention_mask = llm.assemble_inputs(graph_embeds=None, prompt_embeds=input_embeds)
        token_ids = llm.generate_text(
            input_embeds=input_embeds,
            attention_mask=attention_mask,
        )

        text = tokfam.tokenizer.decode(token_ids, skip_special_tokens=True)
        o_texts.append({
            "id": sample.sample_id,
            "rung": sample.rung,
            "llm_output": text,
            "prompt": sample.prompt,
            "query_type": sample.query_type,
            "reasoning": sample.reasoning,
            "formal_form": sample.formal_form
        })
    o_texts = pd.DataFrame(o_texts)
    o_texts.to_csv(f"./ablate_results/output/{cls}_{safe_mdl_name}_mmd-{int(mmd_val)}_qual.csv")


## Baselines (simplest, no cross-family matters)
for mdl in do_models:
    mdl: str = mdl
    print(f"Establishing baseline for LLM {mdl}")
    with LLMWrapper(mdl) as llm:
        mmd_val = 0.0  # MMD doesn't apply here as the projectors aren't involved.
        s2c.w_mmd = mmd_val
        evals2c.w_mmd = mmd_val
        tokfam = TokenizerFamily.from_llm(
            llm,
            max_seq_len=128,
            **arch.family_kwargs()
        )
        blt = BaselineLLMTrainer(tokfam.gate_classifier, device="cuda")

        history = blt.train(s2c, tokfam, train_dataloader=train_d, val_dataloader=val_d, llm=llm)

        yes_ids, no_ids = llm.get_yn_token_sets(tokfam.tokenizer)
        metrics, per_sample = blt._eval(
            val_dataloader=complete_val,
            config=evals2c,
            family=tokfam,
            yes_ids=yes_ids,
            no_ids=no_ids,
            llm=llm,
        )
        eval_and_save(
            llm=llm,
            mmd_val=mmd_val,
            tokfam=tokfam,
            history=history,
            metrics=metrics,
            per_sample=per_sample,
            cls="baseline"
        )










Establishing baseline for LLM Qwen/Qwen3-4B


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

GPU: 7.55GB allocated / 7.55GB reserved
Baseline training (no graph) for Qwen/Qwen3-4B:
[yes] [Yes] [YES] [ yes] [ Yes] [ YES] 
[no] [No] [NO] [ no] [ No] [ NO] 
Epoch    0 | L_gate: 1.222738 | Val Acc: 0.5600 | 0.00s
Epoch    1 | L_gate: 1.092926 | Val Acc: 0.5533 | 0.00s
Epoch    2 | L_gate: 1.180660 | Val Acc: 0.5067 | 0.00s


KeyboardInterrupt: 

In [ ]:
import gc
del blt
del tokfam
llm.unload()
del llm
flush_gpu()
gc.collect()


raise RuntimeError("Stop here for now....")
encoder_ablations = ["cadgn", "adgn", "gcn", "gat", "dec"]
# dec = replace CADGNDecoder with Identity() (to see what happens)

